# Riscrittura codice Cleaning_2 sostituendo l'excel 

Caricamento librerie e file

In [4]:
from __future__ import annotations
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
from config import DatasetConfig, ADNIMERGE

ADNIMERGE.source = "ADNIMERGE_cleaned_01.csv"
OUTPUT_FILE = "ADNIMERGE_cleaned_02.csv"
CUTOFFS_FILE = "cutoffs.json"

Funzione da richiamare per salvare le modifiche su ADNIMERGE_cleaned_02.csv

In [11]:
def save_dataset(df, path):
   df.to_csv(path, index=False)
   print(f"Salvato: {path}  (shape: {df.shape})")

## Rimozione colonne con troppi valori non validi per l'analizi
Funzione di pulizia colonne. Calcola, per ogni colonna, la percentuale di valori validi e scarta quelle sotto la soglia definita in config.MISSING_KEEP_THRESHOLD.

In [59]:
def remove_param_few_subjects(df, threshold=config.MISSING_KEEP_THRESHOLD):
    df = df.copy()
    valid_ratio = df.notna().mean()
    dropped = valid_ratio[valid_ratio < threshold].index.tolist()
    df = df.drop(columns=dropped)
    return df, dropped

Esecuzione dello step. Legge il file originale, applica la pulizia, stampa quali colonne sono state scartate e la variazione di shape, poi salva il risultato con save_dataset().

In [60]:
df_raw = pd.read_csv(ADNIMERGE.source)

df_cleaned, dropped_columns = remove_param_few_subjects(df_raw)
 
print(f"[STEP 1] Colonne scartate ({len(dropped_columns)}): {dropped_columns}")
print(f"[STEP 1] Shape: {df_raw.shape} -> {df_cleaned.shape}")
 
save_dataset(df_cleaned, OUTPUT_FILE)  

C:\Users\ifabb\AppData\Local\Temp\ipykernel_26296\914369842.py:1: DtypeWarning: Columns (0: TTAU_CSF, 1: TAU_bl, 2: PTAU_bl) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv(ADNIMERGE.source)


[STEP 1] Colonne scartate (46): ['FDG', 'PIB', 'AV45', 'FBB', 'AB42_CSF', 'TTAU_CSF', 'PT181_CSF', 'DIGITSCOR', 'MOCA', 'EcogPtMem', 'EcogPtLang', 'EcogPtVisspat', 'EcogPtPlan', 'EcogPtOrgan', 'EcogPtDivatt', 'EcogPtTotal', 'EcogSPMem', 'EcogSPLang', 'EcogSPVisspat', 'EcogSPPlan', 'EcogSPOrgan', 'EcogSPDivatt', 'EcogSPTotal', 'FLDSTRENG', 'DIGITSCOR_bl', 'MOCA_bl', 'EcogPtMem_bl', 'EcogPtLang_bl', 'EcogPtVisspat_bl', 'EcogPtPlan_bl', 'EcogPtOrgan_bl', 'EcogPtDivatt_bl', 'EcogPtTotal_bl', 'EcogSPMem_bl', 'EcogSPLang_bl', 'EcogSPVisspat_bl', 'EcogSPPlan_bl', 'EcogSPOrgan_bl', 'EcogSPDivatt_bl', 'EcogSPTotal_bl', 'ABETA_bl', 'TAU_bl', 'PTAU_bl', 'PIB_bl', 'AV45_bl', 'FBB_bl']
[STEP 1] Shape: (11458, 118) -> (11458, 72)
Salvato: ADNIMERGE_cleaned_02.csv  (shape: (11458, 72))


Verifica di coerenza. Rilegge il file appena scritto e controlla che corrisponda davvero a df_cleaned, per evitare di proseguire con un file non aggiornato (il problema riscontrato in precedenza).

In [61]:
check_step1 = pd.read_csv(OUTPUT_FILE)
assert check_step1.shape == df_cleaned.shape, "[STEP 1] File salvato NON corrisponde a df_cleaned!"
print(f"[STEP 1] Verifica OK: '{OUTPUT_FILE}' ha shape {check_step1.shape}")

[STEP 1] Verifica OK: 'ADNIMERGE_cleaned_02.csv' ha shape (11458, 72)


## Conversione in dummy delle colonne 'GENDER', 'MARRY', 'ETHNICITY', 'RACE', 'DX'
Configurazione dello step. Definisce input/output (il file appena prodotto dallo STEP 1) e l'elenco delle variabili categoriche da convertire.

In [62]:
DUMMY_INPUT = OUTPUT_FILE             # <-- usa direttamente l'output dello step 1
DUMMY_OUTPUT = "ADNIMERGE_cleaned_02.csv"

REF_LIST = ['GENDER', 'MARRY', 'ETHNICITY', 'RACE', 'DX']

Funzione di conversione in dummy. Individua tra ref_list le colonne presenti nel dataframe e le trasforma in variabili binarie con pd.get_dummies.

In [63]:
def classes_to_dummies(df, ref_list):
    df = df.copy()
    to_dummy_list = [c for c in ref_list if c in df.columns]
    if to_dummy_list:
        df = pd.get_dummies(df, columns=to_dummy_list, dtype=int)
    return df, to_dummy_list

Funzione di conteggio dummy. Conta quante colonne dummy sono state create in totale e quante per ciascuna variabile originale.

In [64]:
def count_dummy_columns(original_df, final_df, converted_columns):
    new_dummy_columns = [c for c in final_df.columns if c not in original_df.columns]
    per_column_count = {
        col: len([c for c in new_dummy_columns if c.startswith(col + "_")])
        for col in converted_columns
    }
    return len(new_dummy_columns), per_column_count

Controllo di sicurezza pre-esecuzione. Legge il file d'ingresso e verifica che abbia già il numero di colonne atteso dallo STEP 1 (72), bloccando l'esecuzione con un messaggio chiaro se non è così.

In [65]:
df_step2 = pd.read_csv(DUMMY_INPUT)

In [66]:
assert df_step2.shape[1] == df_cleaned.shape[1], (
    f"[STEP 2] Attenzione: '{DUMMY_INPUT}' ha {df_step2.shape[1]} colonne, "
    f"attese {df_cleaned.shape[1]}. Rieseguire lo STEP 1 prima di procedere."
)

Esecuzione e salvataggio. Applica la conversione in dummy e salva il risultato con save_dataset().

In [67]:
final_df, converted_columns = classes_to_dummies(df_step2, ref_list=REF_LIST)

save_dataset(final_df, DUMMY_OUTPUT)

Salvato: ADNIMERGE_cleaned_02.csv  (shape: (11458, 84))


### Report finale. Riepiloga colonne convertite, variazione di shape e conteggio dettagliato delle dummy create per ciascuna variabile.

In [68]:
total_dummy_count, per_column_count = count_dummy_columns(df_step2, final_df, converted_columns)
print(f"[STEP 2] Colonne convertite in dummy: {converted_columns}")
print(f"[STEP 2] Shape: {df_step2.shape} -> {final_df.shape}")
print(f"[STEP 2] Totale colonne dummy create: {total_dummy_count}")
for col, n in per_column_count.items():
    print(f"  '{col}' -> {n} colonne dummy")

[STEP 2] Colonne convertite in dummy: ['GENDER', 'MARRY', 'ETHNICITY', 'RACE', 'DX']
[STEP 2] Shape: (11458, 72) -> (11458, 84)
[STEP 2] Totale colonne dummy create: 17
  'GENDER' -> 2 colonne dummy
  'MARRY' -> 4 colonne dummy
  'ETHNICITY' -> 2 colonne dummy
  'RACE' -> 6 colonne dummy
  'DX' -> 3 colonne dummy


Controllo modifica andata a buon fine.

In [69]:
pd.read_csv("ADNIMERGE_cleaned_02.csv").head(5)

,RID,COLPROT,ORIGPROT,PTID,SITE,VISCODE,EXAMDATE,DX_bl,AGE_bl,EDUCATION,...,ETHNICITY_1.0,RACE_0.0,RACE_1.0,RACE_2.0,RACE_3.0,RACE_4.0,RACE_5.0,DX_0,DX_1,DX_2
0,2,ADNI1,ADNI1,011_S_0002,11,bl,2005-09-08,CN,74.3,16,...,0,0,0,0,0,0,1,1,0,0
1,2,ADNI1,ADNI1,011_S_0002,11,m06,2006-03-06,CN,74.3,16,...,0,0,0,0,0,0,1,1,0,0
2,2,ADNI1,ADNI1,011_S_0002,11,m36,2008-08-27,CN,74.3,16,...,0,0,0,0,0,0,1,1,0,0
3,2,ADNIGO,ADNI1,011_S_0002,11,m60,2010-09-22,CN,74.3,16,...,0,0,0,0,0,0,1,1,0,0
4,2,ADNI2,ADNI1,011_S_0002,11,m72,2011-09-19,CN,74.3,16,...,0,0,0,0,0,0,1,1,0,0


Controllo solo le colonne aggiunt dummy.

In [70]:
final_df[[c for c in final_df.columns if c not in df_step2.columns]].tail(5)

,GENDER_0,GENDER_1,MARRY_0.0,MARRY_1.0,MARRY_2.0,MARRY_3.0,ETHNICITY_0.0,ETHNICITY_1.0,RACE_0.0,RACE_1.0,RACE_2.0,RACE_3.0,RACE_4.0,RACE_5.0,DX_0,DX_1,DX_2
11453,0,1,0,1,0,0,0,1,1,0,0,0,0,0,0,1,0
11454,1,0,0,0,1,0,1,0,0,0,0,0,1,0,1,0,0
11455,1,0,1,0,0,0,1,0,0,0,0,0,1,0,0,1,0
11456,0,1,0,1,0,0,1,0,0,0,0,0,0,1,0,1,0
11457,1,0,0,1,0,0,0,1,1,0,0,0,0,0,1,0,0


## Rimozione righe in cui tutte le colonne chiave sono vuote

Colonne chiave: misure volumetriche cerebrali fondamentali

In [71]:
KEY_COLUMNS = ['Ventricles', 'Hippocampus', 'WholeBrain', 'ICV']

Funzione di filtro sulle righe

In [72]:
def drop_if_all_none(df, key_columns):
    df = df.copy()
    cols_present = [c for c in key_columns if c in df.columns]
    mask_all_none = df[cols_present].isna().all(axis=1)
    dropped_rows = int(mask_all_none.sum())
    df = df[~mask_all_none]
    return df, dropped_rows

Caricamento e applicazione del filtro

In [73]:
df_step3 = pd.read_csv(OUTPUT_FILE)
df_final, dropped_rows = drop_if_all_none(df_step3, KEY_COLUMNS)

Controllo

In [74]:
print(f"[STEP 3] Colonne chiave verificate: {KEY_COLUMNS}")
print(f"[STEP 3] Righe scartate (tutte le colonne chiave nulle): {dropped_rows}")
print(f"[STEP 3] Shape: {df_step3.shape} -> {df_final.shape}")

[STEP 3] Colonne chiave verificate: ['Ventricles', 'Hippocampus', 'WholeBrain', 'ICV']
[STEP 3] Righe scartate (tutte le colonne chiave nulle): 2152
[STEP 3] Shape: (11458, 84) -> (9306, 84)


In [75]:
save_dataset(df_final, OUTPUT_FILE)

Salvato: ADNIMERGE_cleaned_02.csv  (shape: (9306, 84))


### Statistiche sui soggetti: totale e visite multiple

In [76]:
def count_unique_subjects(df, id_column='RID'):
    """Conta il numero totale di soggetti unici nel dataset."""
    tot_sub = len(df[id_column].unique())
    return tot_sub

In [77]:
def count_multiple_visits(df, id_column='RID'):
    """Conta quanti soggetti hanno più di una visita."""
    multiple_visits = (df[id_column].value_counts() > 1).sum()
    return multiple_visits

In [78]:
final_df = pd.read_csv(OUTPUT_FILE)


C:\Users\ifabb\AppData\Local\Temp\ipykernel_26296\4233954804.py:1: DtypeWarning: Columns (0: FLDSTRENG_bl) have mixed types. Specify dtype option on import or set low_memory=False.
  final_df = pd.read_csv(OUTPUT_FILE)


In [79]:
tot_sub = count_unique_subjects(final_df)
print('totale subject:', tot_sub)

totale subject: 2348


In [80]:
multiple_visits = count_multiple_visits(final_df)
print('subjects with multiple visits: ', multiple_visits)

subjects with multiple visits:  1930


## Riallineamento types delle colonne

Funzione di riallineamento che controllo ogni colonna

In [81]:
def realign_column_types(df):
    """Ispeziona ogni colonna e la riallinea al tipo corretto (int, float o testo)."""
    df = df.copy()
    type_report = {}

    for col in df.columns:
        original_dtype = str(df[col].dtype)

        # Prova la conversione numerica; se fallisce, la colonna resta testo
        converted = pd.to_numeric(df[col], errors='coerce')
        is_fully_numeric = converted.notna().sum() == df[col].notna().sum()

        if is_fully_numeric:
            if (converted.dropna() % 1 == 0).all():
                df[col] = converted.astype('Int64')   # int nullable, gestisce i NaN
                new_dtype = 'Int64'
            else:
                df[col] = converted.astype('float64')
                new_dtype = 'float64'
        else:
            df[col] = df[col].astype(str).where(df[col].notna(), np.nan)
            new_dtype = 'str'

        if original_dtype != new_dtype:
            type_report[col] = (original_dtype, new_dtype)

    return df, type_report


In [82]:
final_df = pd.read_csv(OUTPUT_FILE)
final_df, type_report = realign_column_types(final_df)

C:\Users\ifabb\AppData\Local\Temp\ipykernel_26296\438724613.py:1: DtypeWarning: Columns (0: FLDSTRENG_bl) have mixed types. Specify dtype option on import or set low_memory=False.
  final_df = pd.read_csv(OUTPUT_FILE)


Controllo

In [83]:
print(f"[STEP4] Colonne con tipo riallineato: {len(type_report)}")
for col, (old, new) in type_report.items():
    print(f"  '{col}': {old} -> {new}")

[STEP4] Colonne con tipo riallineato: 51
  'RID': int64 -> Int64
  'SITE': int64 -> Int64
  'EDUCATION': int64 -> Int64
  'APOE4': float64 -> Int64
  'ADASQ4': float64 -> Int64
  'MMSE': float64 -> Int64
  'RAVLT_immediate': float64 -> Int64
  'RAVLT_learning': float64 -> Int64
  'RAVLT_forgetting': float64 -> Int64
  'LDELTOTAL': float64 -> Int64
  'FAQ': float64 -> Int64
  'IMAGEUID': float64 -> Int64
  'WholeBrain': float64 -> Int64
  'Entorhinal': float64 -> Int64
  'Fusiform': float64 -> Int64
  'MidTemp': float64 -> Int64
  'ICV': float64 -> Int64
  'ADASQ4_bl': float64 -> Int64
  'MMSE_bl': float64 -> Int64
  'RAVLT_immediate_bl': float64 -> Int64
  'RAVLT_learning_bl': float64 -> Int64
  'RAVLT_forgetting_bl': float64 -> Int64
  'LDELTOTAL_BL': float64 -> Int64
  'TRABSCOR_bl': float64 -> Int64
  'FAQ_bl': float64 -> Int64
  'IMAGEUID_bl': float64 -> Int64
  'WholeBrain_bl': float64 -> Int64
  'Entorhinal_bl': float64 -> Int64
  'Fusiform_bl': float64 -> Int64
  'MidTemp_bl': f

In [84]:
save_dataset(final_df, OUTPUT_FILE)

Salvato: ADNIMERGE_cleaned_02.csv  (shape: (9306, 84))


## Visualizzaszione degli Outliers + conteggio per ogni colonna analizzata

In [85]:
COLUMNS = [
    'APOE4', 'CDRSB', 'ADAS11', 'ADAS13', 'ADASQ4', 'MMSE', 'RAVLT_immediate',
    'RAVLT_learning', 'RAVLT_forgetting', 'RAVLT_perc_forgetting', 'LDELTOTAL',
    'TRABSCOR', 'FAQ', 'FSVERSION', 'IMAGEUID', 'Ventricles', 'Hippocampus',
    'WholeBrain', 'Entorhinal', 'Fusiform', 'MidTemp', 'ICV', 'mPACCdigit',
    'mPACCtrailsB', 'EXAMDATE_bl', 'CDRSB_bl', 'ADAS11_bl', 'ADAS13_bl',
    'ADASQ4_bl', 'MMSE_bl', 'RAVLT_immediate_bl', 'RAVLT_learning_bl',
    'RAVLT_forgetting_bl', 'RAVLT_perc_forgetting_bl', 'LDELTOTAL_BL',
    'TRABSCOR_bl', 'FAQ_bl', 'mPACCdigit_bl', 'mPACCtrailsB_bl', 'FLDSTRENG_bl',
    'FSVERSION_bl', 'IMAGEUID_bl', 'Ventricles_bl', 'Hippocampus_bl',
    'WholeBrain_bl', 'Entorhinal_bl', 'Fusiform_bl', 'MidTemp_bl', 'ICV_bl',
    'FDG_bl'
]

Conta gli outlier di una colonna numerica con il metodo IQR (1.5 * IQR oltre Q1/Q3).

In [86]:
def count_outliers_iqr(series):
    data = series.dropna()
    q1, q3 = data.quantile(0.25), data.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    n_outliers = int(((data < lower_bound) | (data > upper_bound)).sum())
    return n_outliers

Calcola gli outlier IQR per l'elenco di colonne fornito, senza generare grafici.
    Le colonne non numeriche o assenti vengono segnalate a parte.

In [87]:
def count_outliers_for_columns(df, columns):
    df = df.copy()
    outlier_counts = {}
    skipped_columns = {}

    for col in columns:
        if col not in df.columns:
            skipped_columns[col] = "assente nel dataset"
            continue
        if not pd.api.types.is_numeric_dtype(df[col]):
            skipped_columns[col] = "non numerica"
            continue
        outlier_counts[col] = count_outliers_iqr(df[col])

    return outlier_counts, skipped_columns

In [88]:
df = pd.read_csv(OUTPUT_FILE, low_memory=False)
outlier_counts, skipped_columns = count_outliers_for_columns(df, COLUMNS)

In [89]:
print(f"[OUTLIER] Colonne analizzate: {len(COLUMNS)}")
print(f"[OUTLIER] Colonne numeriche valutate: {len(outlier_counts)}")
print(f"[OUTLIER] Colonne saltate: {len(skipped_columns)}\n")

[OUTLIER] Colonne analizzate: 50
[OUTLIER] Colonne numeriche valutate: 47
[OUTLIER] Colonne saltate: 3



In [90]:
print("[OUTLIER] Numero di outlier per colonna (metodo IQR, 1.5*IQR):")
for col, n in outlier_counts.items():
    print(f"  '{col}': {n} outlier")

[OUTLIER] Numero di outlier per colonna (metodo IQR, 1.5*IQR):
  'APOE4': 0 outlier
  'CDRSB': 469 outlier
  'ADAS11': 394 outlier
  'ADAS13': 216 outlier
  'ADASQ4': 0 outlier
  'MMSE': 329 outlier
  'RAVLT_immediate': 11 outlier
  'RAVLT_learning': 13 outlier
  'RAVLT_forgetting': 189 outlier
  'RAVLT_perc_forgetting': 16 outlier
  'LDELTOTAL': 0 outlier
  'TRABSCOR': 990 outlier
  'FAQ': 845 outlier
  'FSVERSION': 0 outlier
  'IMAGEUID': 1183 outlier
  'Ventricles': 300 outlier
  'Hippocampus': 38 outlier
  'WholeBrain': 46 outlier
  'Entorhinal': 55 outlier
  'Fusiform': 68 outlier
  'MidTemp': 85 outlier
  'ICV': 87 outlier
  'mPACCdigit': 127 outlier
  'mPACCtrailsB': 157 outlier
  'CDRSB_bl': 248 outlier
  'ADAS11_bl': 318 outlier
  'ADAS13_bl': 166 outlier
  'ADASQ4_bl': 0 outlier
  'MMSE_bl': 289 outlier
  'RAVLT_immediate_bl': 1 outlier
  'RAVLT_learning_bl': 0 outlier
  'RAVLT_forgetting_bl': 200 outlier
  'RAVLT_perc_forgetting_bl': 10 outlier
  'LDELTOTAL_BL': 0 outlier
  

In [91]:
if skipped_columns:
    print("\n[OUTLIER] Colonne saltate (non numeriche o assenti):")
    for col, reason in skipped_columns.items():
        print(f"  '{col}': {reason}")


[OUTLIER] Colonne saltate (non numeriche o assenti):
  'EXAMDATE_bl': non numerica
  'FLDSTRENG_bl': non numerica
  'FSVERSION_bl': non numerica


In [92]:
print(f"\n[OUTLIER] Totale outlier rilevati: {sum(outlier_counts.values())}")


[OUTLIER] Totale outlier rilevati: 10773


## Visualizzazione degli Outliers con la Heatmap
Parole chiave che identificano colonne da escludere: id, date, età, dati demografici

In [93]:
EXCLUDE_KEYWORDS = [
    'RID', 'PTID', 'COLPROT', 'ORIGPROT', 'SITE', 'VISCODE', 'EXAMDATE',
    'AGE', 'GENDER', 'MARRY', 'ETHNICITY', 'RACE', 'DX', 'EDUCATION',
    'APOE4', 'FSVERSION', 'FLDSTRENG', 'IMAGEUID', 'update_stamp',
    'Month', 'Years_bl', 'VISIT_MONTH', '_bl'
]
EXCLUDE_EXACT = ['M']  # colonne da escludere per nome esatto (es. 'M' = mesi dalla baseline)

Seleziona solo le colonne numeriche di misurazione (volumetriche/cliniche),
    scartando id, date, età e variabili demografiche (dummy incluse).

In [94]:
def select_measurement_columns(df, exclude_keywords=EXCLUDE_KEYWORDS, exclude_exact=EXCLUDE_EXACT):
    df = df.copy()
    kept_columns = []
    excluded_columns = []

    for col in df.columns:
        is_excluded = (
            col in exclude_exact
            or any(kw.lower() in col.lower() for kw in exclude_keywords)
        )
        is_numeric = pd.api.types.is_numeric_dtype(df[col])

        if is_excluded or not is_numeric:
            excluded_columns.append(col)
        else:
            kept_columns.append(col)

    return kept_columns, excluded_columns

Conta gli outlier di una colonna numerica con il metodo IQR (1.5 * IQR oltre Q1/Q3).

In [95]:
def count_outliers_iqr(series):
    data = series.dropna()
    q1, q3 = data.quantile(0.25), data.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    n_outliers = int(((data < lower_bound) | (data > upper_bound)).sum())
    return n_outliers

Genera una heatmap con il numero e la percentuale di outlier (metodo IQR)
    per ciascuna colonna di misurazione.

In [96]:
def plot_outliers_heatmap(df, columns, title, output_path):
    counts = []
    percentages = []
    for col in columns:
        n_outliers = count_outliers_iqr(df[col])
        n_valid = df[col].notna().sum()
        pct = (n_outliers / n_valid * 100) if n_valid > 0 else 0
        counts.append(n_outliers)
        percentages.append(pct)

    outlier_counts = dict(zip(columns, counts))

    data = np.array(percentages).reshape(1, -1)

    fig, ax = plt.subplots(figsize=(max(10, len(columns) * 0.7), 3))
    im = ax.imshow(data, cmap='Reds', aspect='auto')

    ax.set_xticks(range(len(columns)))
    ax.set_xticklabels(columns, rotation=45, ha='right', fontsize=9)
    ax.set_yticks([])

    for i, (n, pct) in enumerate(zip(counts, percentages)):
        ax.text(i, 0, f"{n}\n({pct:.1f}%)", ha='center', va='center', fontsize=8)

    fig.colorbar(im, ax=ax, label='% outlier sul totale valori validi', orientation='horizontal', pad=0.35)
    ax.set_title(title, fontsize=13)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)

    return outlier_counts

Legge il dataset, seleziona le colonne di misurazione, calcola e visualizza gli outlier

In [97]:
df = pd.read_csv(OUTPUT_FILE, low_memory=False)
kept_columns, excluded_columns = select_measurement_columns(df)

In [98]:
print(f"Colonne totali: {len(df.columns)}")
print(f"Colonne di misurazione mantenute ({len(kept_columns)}): {kept_columns}")

Colonne totali: 84
Colonne di misurazione mantenute (21): ['CDRSB', 'ADAS11', 'ADAS13', 'ADASQ4', 'MMSE', 'RAVLT_immediate', 'RAVLT_learning', 'RAVLT_forgetting', 'RAVLT_perc_forgetting', 'LDELTOTAL', 'TRABSCOR', 'FAQ', 'Ventricles', 'Hippocampus', 'WholeBrain', 'Entorhinal', 'Fusiform', 'MidTemp', 'ICV', 'mPACCdigit', 'mPACCtrailsB']


In [99]:
outlier_counts = plot_outliers_heatmap(
    df, kept_columns,
    title="Outlier delle variabili di visita (volumetriche/cliniche) - ADNIMERGE (metodo IQR)",
    output_path="adnimerge_outliers_heatmap.png"
)
print(f"Heatmap salvata in: adnimerge_outliers_heatmap.png")

Heatmap salvata in: adnimerge_outliers_heatmap.png


Log finale: numero di outlier per colonna e totale complessivo

In [100]:
print("\n[OUTLIER] Numero di outlier per colonna (metodo IQR, 1.5*IQR):")
for col, n in outlier_counts.items():
    print(f"  '{col}': {n} outlier")
print(f"\n[OUTLIER] Totale outlier rilevati: {sum(outlier_counts.values())}")


[OUTLIER] Numero di outlier per colonna (metodo IQR, 1.5*IQR):
  'CDRSB': 469 outlier
  'ADAS11': 394 outlier
  'ADAS13': 216 outlier
  'ADASQ4': 0 outlier
  'MMSE': 329 outlier
  'RAVLT_immediate': 11 outlier
  'RAVLT_learning': 13 outlier
  'RAVLT_forgetting': 189 outlier
  'RAVLT_perc_forgetting': 16 outlier
  'LDELTOTAL': 0 outlier
  'TRABSCOR': 990 outlier
  'FAQ': 845 outlier
  'Ventricles': 300 outlier
  'Hippocampus': 38 outlier
  'WholeBrain': 46 outlier
  'Entorhinal': 55 outlier
  'Fusiform': 68 outlier
  'MidTemp': 85 outlier
  'ICV': 87 outlier
  'mPACCdigit': 127 outlier
  'mPACCtrailsB': 157 outlier

[OUTLIER] Totale outlier rilevati: 4435


## Controllo incrociato tra le mie variabili e quelle segnalate da Chiara del csv statistics
Elenco di riferimento fornito da confrontare con le colonne effettive del dataset

In [6]:
COLONNE_CHIARA = [
    'RID', 'COLPROT', 'VISCODE', 'EXAMDATE', 'AGE', 'PTGENDER', 'PTEDUCAT',
    'PTETHCAT', 'PTRACCAT', 'PTMARRY', 'APOE4', 'CDRSB', 'ADAS11', 'ADAS13',
    'MMSE', 'RAVLT_immediate', 'FAQ', 'MOCA', 'FLDSTRENG', 'FSVERSION',
    'IMAGEUID', 'Ventricles', 'Hippocampus', 'Entorhinal', 'Fusiform',
    'MidTemp', 'ICV', 'DX', 'update_stamp'
]

Confronta le colonne del dataset con un elenco di riferimento,
    in entrambe le direzioni.

In [102]:
def compare_columns(df, reference_columns):
    dataset_columns = list(df.columns)

    not_in_reference = [c for c in dataset_columns if c not in reference_columns]
    not_in_dataset = [c for c in reference_columns if c not in dataset_columns]

    return not_in_reference, not_in_dataset


In [103]:
df = pd.read_csv(OUTPUT_FILE, low_memory=False)
not_in_reference, not_in_dataset = compare_columns(df, COLONNE_CHIARA)

In [104]:
print(f"Colonne totali nel dataset: {len(df.columns)}")
print(f"Colonne in COLONNE_CHIARA: {len(COLONNE_CHIARA)}\n")

Colonne totali nel dataset: 84
Colonne in COLONNE_CHIARA: 29



In [105]:
print(f"Colonne del dataset NON presenti in COLONNE_CHIARA ({len(not_in_reference)}):")
for c in not_in_reference:
    print(f"  - {c}")

Colonne del dataset NON presenti in COLONNE_CHIARA (63):
  - ORIGPROT
  - PTID
  - SITE
  - DX_bl
  - AGE_bl
  - EDUCATION
  - ADASQ4
  - RAVLT_learning
  - RAVLT_forgetting
  - RAVLT_perc_forgetting
  - LDELTOTAL
  - TRABSCOR
  - WholeBrain
  - mPACCdigit
  - mPACCtrailsB
  - EXAMDATE_bl
  - CDRSB_bl
  - ADAS11_bl
  - ADAS13_bl
  - ADASQ4_bl
  - MMSE_bl
  - RAVLT_immediate_bl
  - RAVLT_learning_bl
  - RAVLT_forgetting_bl
  - RAVLT_perc_forgetting_bl
  - LDELTOTAL_BL
  - TRABSCOR_bl
  - FAQ_bl
  - mPACCdigit_bl
  - mPACCtrailsB_bl
  - FLDSTRENG_bl
  - FSVERSION_bl
  - IMAGEUID_bl
  - Ventricles_bl
  - Hippocampus_bl
  - WholeBrain_bl
  - Entorhinal_bl
  - Fusiform_bl
  - MidTemp_bl
  - ICV_bl
  - FDG_bl
  - Years_bl
  - Month_bl
  - Month
  - M
  - VISIT_MONTH
  - GENDER_0
  - GENDER_1
  - MARRY_0.0
  - MARRY_1.0
  - MARRY_2.0
  - MARRY_3.0
  - ETHNICITY_0.0
  - ETHNICITY_1.0
  - RACE_0.0
  - RACE_1.0
  - RACE_2.0
  - RACE_3.0
  - RACE_4.0
  - RACE_5.0
  - DX_0
  - DX_1
  - DX_2


In [106]:
print(f"\nColonne di COLONNE_CHIARA NON presenti nel dataset ({len(not_in_dataset)}):")
for c in not_in_dataset:
    print(f"  - {c}")


Colonne di COLONNE_CHIARA NON presenti nel dataset (8):
  - PTGENDER
  - PTEDUCAT
  - PTETHCAT
  - PTRACCAT
  - PTMARRY
  - MOCA
  - FLDSTRENG
  - DX


## Mantiene solo le colonne presenti in COLONNE_CHIARA
Funzione di selezione colonne.

In [ ]:
# def keep_only_reference_columns(df, reference_columns):
#     df = df.copy()
#     cols_to_keep = [c for c in reference_columns if c in df.columns]
#     missing_in_dataset = [c for c in reference_columns if c not in df.columns]
#     dropped_columns = [c for c in df.columns if c not in reference_columns]
#     df = df[cols_to_keep]
#     return df, dropped_columns, missing_in_dataset

Caricamento, applicazione del filtro e report a schermo

In [ ]:
# df_step_chiara = pd.read_csv(OUTPUT_FILE, low_memory=False)
# df_chiara, dropped_columns, missing_in_dataset = keep_only_reference_columns(df_step_chiara, COLONNE_CHIARA)

In [ ]:
# print(f"[STEP CHIARA] Colonne mantenute: {len(df_chiara.columns)} / {len(COLONNE_CHIARA)}")
# print(f"[STEP CHIARA] Colonne scartate dal dataset ({len(dropped_columns)}):")
# for c in dropped_columns:
#     print(f"  - {c}")
# if missing_in_dataset:
#     print(f"\n[STEP CHIARA] Attenzione: colonne di COLONNE_CHIARA non trovate nel dataset ({len(missing_in_dataset)}):")
#     for c in missing_in_dataset:
#         print(f"  - {c}")
# print(f"\n[STEP CHIARA] Shape: {df_step_chiara.shape} -> {df_chiara.shape}")

[STEP CHIARA] Colonne mantenute: 21 / 29
[STEP CHIARA] Colonne scartate dal dataset (63):
  - ORIGPROT
  - PTID
  - SITE
  - DX_bl
  - AGE_bl
  - EDUCATION
  - ADASQ4
  - RAVLT_learning
  - RAVLT_forgetting
  - RAVLT_perc_forgetting
  - LDELTOTAL
  - TRABSCOR
  - WholeBrain
  - mPACCdigit
  - mPACCtrailsB
  - EXAMDATE_bl
  - CDRSB_bl
  - ADAS11_bl
  - ADAS13_bl
  - ADASQ4_bl
  - MMSE_bl
  - RAVLT_immediate_bl
  - RAVLT_learning_bl
  - RAVLT_forgetting_bl
  - RAVLT_perc_forgetting_bl
  - LDELTOTAL_BL
  - TRABSCOR_bl
  - FAQ_bl
  - mPACCdigit_bl
  - mPACCtrailsB_bl
  - FLDSTRENG_bl
  - FSVERSION_bl
  - IMAGEUID_bl
  - Ventricles_bl
  - Hippocampus_bl
  - WholeBrain_bl
  - Entorhinal_bl
  - Fusiform_bl
  - MidTemp_bl
  - ICV_bl
  - FDG_bl
  - Years_bl
  - Month_bl
  - Month
  - M
  - VISIT_MONTH
  - GENDER_0
  - GENDER_1
  - MARRY_0.0
  - MARRY_1.0
  - MARRY_2.0
  - MARRY_3.0
  - ETHNICITY_0.0
  - ETHNICITY_1.0
  - RACE_0.0
  - RACE_1.0
  - RACE_2.0
  - RACE_3.0
  - RACE_4.0
  - RACE_5.0


Salvataggio del risultato

In [ ]:
# save_dataset(df_chiara, OUTPUT_FILE)

Salvato: ADNIMERGE_cleaned_02.csv  (shape: (9306, 21))


## Mantiene le colonne presenti in COLONNE_CHIARA(asludendo quelle generate dal codice precedentemente)

Colonne aggiuntive da mantenere comunque (dummy + variabile di visita),
anche se non presenti in COLONNE_CHIARA

In [1]:
COLONNE_EXTRA_DA_MANTENERE = [
    'VISIT_MONTH', 'AGE', 'GENDER_0', 'GENDER_1',
    'MARRY_0.0', 'MARRY_1.0', 'MARRY_2.0', 'MARRY_3.0',
    'ETHNICITY_0.0', 'ETHNICITY_1.0',
    'RACE_0.0', 'RACE_1.0', 'RACE_2.0', 'RACE_3.0', 'RACE_4.0', 'RACE_5.0',
    'DX_0', 'DX_1', 'DX_2'
]

Mantiene solo le colonne presenti in reference_columns (più eventuali
    colonne extra da conservare comunque), scartando tutte le altre

In [2]:
def keep_only_reference_columns(df, reference_columns, extra_columns=None):
    df = df.copy()
    extra_columns = extra_columns or []

    # Unisce l'elenco di riferimento con le colonne extra, senza duplicati
    columns_to_keep_full = list(dict.fromkeys(reference_columns + extra_columns))

    cols_to_keep = [c for c in columns_to_keep_full if c in df.columns]
    missing_in_dataset = [c for c in columns_to_keep_full if c not in df.columns]
    dropped_columns = [c for c in df.columns if c not in columns_to_keep_full]

    df = df[cols_to_keep]
    return df, dropped_columns, missing_in_dataset


In [7]:
df_step_chiara = pd.read_csv(OUTPUT_FILE, low_memory=False)
df_chiara, dropped_columns, missing_in_dataset = keep_only_reference_columns(
    df_step_chiara, COLONNE_CHIARA, COLONNE_EXTRA_DA_MANTENERE
)

In [8]:
print(f"[STEP CHIARA] Colonne mantenute: {len(df_chiara.columns)}")
print(f"[STEP CHIARA] Colonne scartate dal dataset ({len(dropped_columns)}):")
for c in dropped_columns:
    print(f"  - {c}")

[STEP CHIARA] Colonne mantenute: 39
[STEP CHIARA] Colonne scartate dal dataset (45):
  - ORIGPROT
  - PTID
  - SITE
  - DX_bl
  - AGE_bl
  - EDUCATION
  - ADASQ4
  - RAVLT_learning
  - RAVLT_forgetting
  - RAVLT_perc_forgetting
  - LDELTOTAL
  - TRABSCOR
  - WholeBrain
  - mPACCdigit
  - mPACCtrailsB
  - EXAMDATE_bl
  - CDRSB_bl
  - ADAS11_bl
  - ADAS13_bl
  - ADASQ4_bl
  - MMSE_bl
  - RAVLT_immediate_bl
  - RAVLT_learning_bl
  - RAVLT_forgetting_bl
  - RAVLT_perc_forgetting_bl
  - LDELTOTAL_BL
  - TRABSCOR_bl
  - FAQ_bl
  - mPACCdigit_bl
  - mPACCtrailsB_bl
  - FLDSTRENG_bl
  - FSVERSION_bl
  - IMAGEUID_bl
  - Ventricles_bl
  - Hippocampus_bl
  - WholeBrain_bl
  - Entorhinal_bl
  - Fusiform_bl
  - MidTemp_bl
  - ICV_bl
  - FDG_bl
  - Years_bl
  - Month_bl
  - Month
  - M


In [9]:
if missing_in_dataset:
    print(f"\n[STEP CHIARA] Attenzione: colonne richieste non trovate nel dataset ({len(missing_in_dataset)}):")
    for c in missing_in_dataset:
        print(f"  - {c}")

print(f"\n[STEP CHIARA] Shape: {df_step_chiara.shape} -> {df_chiara.shape}")


[STEP CHIARA] Attenzione: colonne richieste non trovate nel dataset (8):
  - PTGENDER
  - PTEDUCAT
  - PTETHCAT
  - PTRACCAT
  - PTMARRY
  - MOCA
  - FLDSTRENG
  - DX

[STEP CHIARA] Shape: (9306, 84) -> (9306, 39)


In [12]:
save_dataset(df_chiara, OUTPUT_FILE)

Salvato: ADNIMERGE_cleaned_02.csv  (shape: (9306, 39))
